In [4]:
!pip install -U langchain


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import config


C:\Users\glyze\PyCharmMiscProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [ ]:
url = 'https://www.datacamp.com/tutorial/how-transformers-work'
loader = WebBaseLoader(url)

raw_documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter()

documents = text_splitter.split_documents(raw_documents)


In [ ]:
embeddings = HuggingFaceEmbeddings()
vectorStore = FAISS.from_documents(documents,embeddings)

In [ ]:
memory = InMemoryChatMessageHistory()

In [ ]:
llm = ChatGroq(
    model='llama-3.1-8b-instant',
    api_key=config.GROQ_API_KEY
)

In [1]:
chat = RunnablePassthrough()

NameError: name 'RunnablePassthrough' is not defined

In [20]:
def retrieve_docs(query):
    docs = vectorStore.similarity_search(query)
    return "\n\n".join([d.page_content for d in docs])

In [ ]:
def build_prompt(context_and_question):
    context, context_and_question = context_and_question

    return f"""
    Use the context below to answer the question.

    Context: {context}
    Question: {context_and_question}
"""

In [21]:
prompt = 'What is transformer?'
def main():
    chain = (
            {'context': RunnableLambda(retrieve_docs), 'question': RunnablePassthrough()} | RunnableLambda(build_prompt) | llm
    )
    return chain.invoke(prompt)

In [ ]:
response = main()
print(response)